# Projeto Final – Storytelling sobre Desastres Naturais (2014—2021)

## Setup

### Imports

In [ ]:
import os
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from typing import List
from dotenv import load_dotenv
from urllib.request import urlopen
from mysql.connector import connect

### Environment Variables

In [ ]:
load_dotenv()

MYSQL_HOST=os.getenv(key="MYSQL_HOST")
MYSQL_PORT=os.getenv(key="MYSQL_PORT")
MYSQL_USER=os.getenv(key="MYSQL_USER")
MYSQL_PASS=os.getenv(key="MYSQL_PASS")
MYSQL_DB=os.getenv(key="MYSQL_DB")

### MySQL Connector

In [ ]:
connector = connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    password=MYSQL_PASS,
    database=MYSQL_DB,
    ssl_disabled=False
)

### DataFrames

In [ ]:
# Used to draw each state acronym inside their
# respective area on the Q2 choropleth map:
brazil_state_coordinates = pd.DataFrame({
    "acronym": [
        "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA",
        "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI", "RJ", "RN",
        "RS", "RO", "RR", "SC", "SP", "SE", "TO"
    ],
    "lat": [
        -9.02, -9.62, 1.41, -3.47, -12.97, -5.20, -15.78, -19.19, -16.64, -5.53,
        -12.64, -20.51, -18.10, -3.79, -7.12, -24.89, -8.28, -8.28, -22.84, -5.81,
        -30.03, -10.83, 2.82, -27.33, -23.55, -10.91, -10.25
    ],
    "lon": [
        -70.81, -35.74, -51.77, -65.10, -38.51, -39.53, -47.93, -40.34, -49.31, -44.30,
        -55.42, -54.54, -44.38, -52.48, -34.86, -51.55, -34.88, -43.68, -43.17, -35.21,
        -51.23, -63.34, -60.67, -48.55, -46.63, -37.07, -48.33
    ]
})

## Criar Gráficos

Q1 – _Quais tipos de desastre predominaram entre 2014 e 2021?_

In [ ]:
q1_query = """
    SELECT
        cobrade AS desastre,
        COUNT(*) AS ocorrencias
    FROM fato_desastre
    WHERE status = 'Reconhecido'
    GROUP BY cobrade
    ORDER BY ocorrencias DESC
    LIMIT 10;
"""


cursor = connector.cursor(dictionary=True)
cursor.execute(q1_query)
rows = cursor.fetchall()


q1_items: List[dict] = []
for row in rows:
    temp: List[str] = row.get("desastre", "").split('-')  # type: ignore (Pylance)

    if len(temp) > 2 and ',' not in temp[2]:
        disaster = temp[2]
    else:
        disaster = temp[1]

    q1_items.append({
        "disaster": disaster,
        "occurrences": row.get("ocorrencias", "")  # type: ignore (Pylance)
    })


q1_df = pd.DataFrame(q1_items)

Gráfico: Barras (horizontal)

In [ ]:
fig = px.bar(
    q1_df.sort_values("occurrences"),
    x="occurrences",
    y="disaster",
    orientation="h",
    text="occurrences",
    color="occurrences",
    color_continuous_scale="Teal"
)

fig.update_traces(textposition="outside")

fig.update_layout(
    title="Desastres Predominantes entre 2014 e 2021",
    title_x=0.5,
    template="plotly_white",
    xaxis_title="Nº de Casos",
    yaxis_title="",
    showlegend=False,
    coloraxis_colorbar=dict(title=""),
    width=1000,
    height=600,
)

fig.show()

Q2 – _Onde os desastres se concentraram geograficamente?_

In [ ]:
q2_query = """
    SELECT
        uf,
        municipio,
        COUNT(*) AS ocorrencias
    FROM fato_desastre
    WHERE status = 'Reconhecido'
    GROUP BY uf, municipio
    ORDER BY ocorrencias DESC;
"""


cursor = connector.cursor(dictionary=True)
cursor.execute(q2_query)
rows = cursor.fetchall()


q2_items = [
    {
        "state": row.get("uf", ""),                 # type: ignore (Pylance)
        "city": row.get("municipio", ""),           # type: ignore (Pylance)
        "occurrences": row.get("ocorrencias", "")   # type: ignore (Pylance)
    }
    for row in rows
]


q2_df = pd.DataFrame(q2_items)

Gráfico: Choropleth (mapa)

In [ ]:
URL = "https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson"

with urlopen(URL) as response:
    brazil_geojson = json.load(response)


fig = px.choropleth(
    (
        q2_df.groupby("state", as_index=False)["occurrences"].sum()
    ),
    geojson=brazil_geojson,
    locations="state",
    featureidkey="properties.sigla",
    color="occurrences",
    color_continuous_scale="tempo",
    projection="mercator"
)

fig.update_geos(
    fitbounds="geojson",
    visible=False
)

fig.update_layout(
    title="Maior Concentração de Desastres",
    title_x=0.5,
    template="plotly_white",
    coloraxis_colorbar=dict(title=""),
    width=900,
    height=900,
    margin=dict(l=0, r=0, t=60, b=0)
)

# Customize layout for all states:
df_other = brazil_state_coordinates[
    brazil_state_coordinates["acronym"] != "MG"
]
fig.add_trace(
    go.Scattergeo(
        lon=df_other["lon"],
        lat=df_other["lat"],
        text=df_other["acronym"],
        mode="text",
        textfont=dict(size=10, color="black"),
        hoverinfo="skip",
        showlegend=False
    )
)

# Customize layout only for "MG":
df_mg = brazil_state_coordinates[
    brazil_state_coordinates["acronym"] == "MG"
]
fig.add_trace(
    go.Scattergeo(
        lon=df_mg["lon"],
        lat=df_mg["lat"],
        text=df_mg["acronym"],
        mode="text",
        textfont=dict(size=10, color="white"),
        hoverinfo="skip",
        showlegend=False
    )
)

fig.show()

Q3 – _Quais tipos de desastre geraram maior impacto humano entre 2014 e 2019 (**pré COVID**)?_

In [ ]:
q3_query = """
    SELECT
        cobrade,
        uf,
        municipio,
        SUM(dh_mortos) AS total_mortes
    FROM fato_desastre fd
    WHERE status = 'Reconhecido' AND ano BETWEEN 2014 and 2019 -- sem COVID
    GROUP BY cobrade, uf, municipio
    ORDER BY total_mortes DESC
    LIMIT 5;
"""


cursor = connector.cursor(dictionary=True)
cursor.execute(q3_query)
rows = cursor.fetchall()


q3_items = [
    {
        "disaster": row.get("cobrade", "").split('-')[1],   # type: ignore (Pylance)
        "state": row.get("uf", ""),                         # type: ignore (Pylance)
        "city": row.get("municipio", ""),                   # type: ignore (Pylance)
        "deaths": row.get("total_mortes", "")               # type: ignore (Pylance)
    }
    for row in rows
]


q3_df = pd.DataFrame(q3_items)

Gráfico: Bolha

In [ ]:
q3_df["location"] = q3_df["city"] + " (" + q3_df["state"] + ")"
q3_df["deaths"] = pd.to_numeric(q3_df["deaths"], errors="coerce")

fig = px.scatter(
    q3_df,
    x="location",
    y="disaster",
    size="deaths",
    color="deaths",
    size_max=80,
    color_continuous_scale="burg"
)

fig.update_traces(
    marker=dict(
        sizemode="area",
        sizeref=2.0 * q3_df["deaths"].max() / (80 ** 2),
        sizemin=25,
        line=dict(width=2, color="black")
    )
)

for _, row in q3_df.iterrows():
    font_color = "white" if row["deaths"] >= 100 else "black"
    fig.add_annotation(
        x=row["location"],
        y=row["disaster"],
        text=f"{int(row['deaths'])}",
        showarrow=False,
        font=dict(size=12, color=font_color)
    )

fig.update_layout(
    title="Desastres com Maior Número de Mortes (pré COVID)",
    title_x=0.5,
    template="plotly_white",
    xaxis_title="Cidade (UF)",
    yaxis_title="",
    coloraxis_colorbar=dict(title=""),
    width=1000,
    height=600,
)

fig.show()

Q3 – _Quais tipos de desastre geraram maior impacto humano entre 2014 e 2019 (**pós COVID**)?_

In [ ]:
q3_covid_query = """
    SELECT
        cobrade,
        uf,
        municipio,
        SUM(dh_mortos) AS total_mortes
    FROM fato_desastre fd
    WHERE status = 'Reconhecido'
    GROUP BY cobrade, uf, municipio
    ORDER BY total_mortes DESC
    LIMIT 5;
"""


cursor = connector.cursor(dictionary=True)
cursor.execute(q3_covid_query)
rows = cursor.fetchall()


q3_covid_items = [
    {
        "disaster": row.get("cobrade", "").split('-')[1],   # type: ignore (Pylance)
        "state": row.get("uf", ""),                         # type: ignore (Pylance)
        "city": row.get("municipio", ""),                   # type: ignore (Pylance)
        "deaths": row.get("total_mortes", "")               # type: ignore (Pylance)
    }
    for row in rows
]


q3_covid_df = pd.DataFrame(q3_covid_items)

Gráfico: Bolha

In [ ]:
q3_covid_df["location"] = q3_covid_df["city"] + " (" + q3_covid_df["state"] + ")"
q3_covid_df["deaths"] = pd.to_numeric(q3_covid_df["deaths"], errors="coerce")

fig = px.scatter(
    q3_covid_df,
    x="location",
    y="disaster",
    size="deaths",
    color="deaths",
    size_max=80,
    color_continuous_scale="burg"
)

fig.update_traces(
    marker=dict(
        sizemode="area",
        sizeref=2.0 * q3_covid_df["deaths"].max() / (80 ** 2),
        sizemin=25,
        line=dict(width=2, color="black")
    )
)

for _, row in q3_covid_df.iterrows():
    font_color = "white" if row["deaths"] >= 30000 else "black"
    fig.add_annotation(
        x=row["location"],
        y=row["disaster"],
        text=f"{int(row['deaths'])}",
        showarrow=False,
        font=dict(size=12, color=font_color)
    )

fig.update_layout(
    title="Desastres com Maior Número de Mortes (pós COVID)",
    title_x=0.5,
    template="plotly_white",
    xaxis_title="Cidade (UF)",
    yaxis_title="",
    coloraxis_colorbar=dict(title=""),
    width=1000,
    height=600,
)

fig.show()